> **VERSÃO ANTERIOR — FASE 9 V4.** Para a ampliação de 600 mil empresas úteis por porte de município e nicho, use a **[Fase 9 V5](https://colab.research.google.com/github/wesleyshoto702-hub/januarioleads/blob/main/notebooks/Januario_Leads_Fase_9_V5_600_Mil_Empresas.ipynb)**. A V4 está preservada apenas para histórico e compatibilidade. **Não execute as duas versões simultaneamente.**

# Januário Leads — Fase 9: captar novas empresas Overture em municípios já cobertos

## Objetivo
A importação anterior atingiu aproximadamente **60 empresas por município**, deixando de fora muitos estabelecimentos. Esta etapa faz uma **nova varredura completa** do Overture em cada cidade, identifica os **IDs empresariais ainda ausentes do banco**, mantém candidatos além dos 100 primeiros resultados brutos e tenta acrescentar **até 40 empresas inéditas** onde já existem 60, chegando a **até 100 por município**.

**Importante:** os 5.570 municípios brasileiros já foram contemplados na malha da primeira etapa; o ganho da Fase 9 será em **empresas novas dentro desses municípios**, e no número de municípios beneficiados. Em municípios com menos de 60 registros, a rotina acrescenta **no máximo 40 novas empresas**, podendo terminar abaixo de 100. Este limite de +40 vale entre sessões, não apenas para uma única execução.

### Proteções e lógica
- Destino **exclusivo**: Januário Leads — HOMOLOGAÇÃO (`prlrxjothrslhswpkoza`). Não altera o buscador nem a produção.
- **Não chama Geoapify nem Google Places**. Usa Overture Maps e limites do município da malha oficial do IBGE.
- Ignora rios, lagos e feições geográficas que não são empresas. Puxa os **ID GERS, nome e coordenadas já existentes** da cidade; rejeita IDs repetidos e também candidatos com mesmo nome normalizado a até 15 metros de uma empresa registrada.
- Examina **todo o arquivo de resultados do Overture**, *antes* de selecionar novos candidatos: evita perder os negócios que ficaram de fora da primeira seleção.
- Guarda apenas grupos limitados de novos candidatos para controlar memória e priorizar informações comerciais como telefone, email, site e Instagram, quando disponíveis.
- Máximo de **40 empresas novas por município na Fase 9**, limitado também a **100 empresas totais por município**. O checkpoint guarda o número original para impedir um segundo lote de +40 após retomada; nenhum registro antigo é atualizado. Em colisão de ID, inserção ignora duplicatas.
- Proteção interna: importação **para em 340 MiB** de tamanho do banco; a cota real Supabase Free não foi ampliada.
- **Retomável e nacional:** tenta um município por vez, alternando entre UFs para ampliar a distribuição nacional; evita reconsulta imediata a municípios sem novas empresas usando a tabela privada `overture_phase9_attempts` (homologação).
- `MAX_MUNICIPIOS_SESSAO = 5572` permite percorrer todos os municípios elegíveis na sessão, mas **o Colab interrompe por tempo, erros e espaço antes disso se necessário**. Uma cidade pode conter menos de 100 empresas reais no Overture.
- O Colab só começa a importar depois de executar a última célula e digitar **`AMPLIAR_100`**. Nunca cole credenciais no chat.

## Instruções
1. Execute as **células 1–3**.
2. Verifique que está utilizando a chave **secreta da homologação** e que sua cota Supabase está dentro do limite.
3. Na **célula 4**, digite `AMPLIAR_100` e insira a chave no campo oculto.
4. Envie apenas o relatório `RELATÓRIO FASE 9` (sem chaves). Para retomar, execute novamente a última célula.

**Nenhuma importação é executada automaticamente ao abrir o notebook.**

### Atualização V2 — diagnóstico e recuperação de erros
- Cada falha agora recebe um **código técnico seguro**, gravado no checkpoint `overture_phase9_attempts.error_code`, sem salvar logs brutos, URLs ou chaves.
- Erros transitórios de rede e indisponibilidade podem receber **uma tentativa extra**, com intervalo. Em falhas de autenticação, cota, acesso negado ou armazenamento, o processo **não insiste**.
- Uma sequência de **três falhas** em municípios/UFs diferentes interrompe preventivamente a sessão. Isso evita o padrão da primeira execução, que parou após 12 erros consecutivos.
- O relatório separa causas, tentativas extras e motivo de parada. **Os 12 erros históricos continuam sem código específico**, pois o notebook anterior não registrava os detalhes.
- Se já tinha aberto a versão antiga do Colab, **reabra este notebook e execute novamente as células 1–3** antes de repetir a célula 4.

### Fase 9 V3 — menos downloads e respeito aos limites da fonte
- Baixa Overture para **até três municípios próximos da mesma UF de uma vez**. Usa centroides calculados a partir dos estabelecimentos existentes; se os polígonos forem muito dispersos, divide automaticamente em downloads individuais.
- Intervalo mínimo de **75 segundos entre inícios** de downloads, com folga variável de até 15 segundos. No máximo **240 downloads por sessão**; tempo e armazenamento podem parar antes.
- Após `fonte_taxa` (429), o Colab bloqueia novas consultas por **seis horas desde o último erro registrado**, inclusive se você reabrir o notebook, e exibe o horário de liberação. Respeita também o intervalo entre sessões.
- Não repete chamadas ao receber 429. Para falhas transitórias de rede, permite somente uma tentativa adicional respeitando o mesmo ritmo.
- A tabela de homologação `overture_phase9_city_centers` é **privada da service_role**, e não altera os leads.
- **Importante:** estes parâmetros são conservadores; não garantem que a fonte deixará de impor limites. O número de registros por município continua em 100 e a parada do banco em 340 MiB.

Abra uma sessão nova do Colab e execute **todas as células 1–3** antes da célula 4 para carregar esta versão.

### Fase 9 V4 — até 40 empresas inéditas por município, inclusive onde havia menos de 60
- Guarda o `before_count` da primeira tentativa como base estável. Reaberturas não acumulam outro lote de 40 por engano.
- A fila pula cidades que já ganharam 40 registros, mesmo quando ainda não chegaram a 100.
- O script faz a mesma validação antes de inserir, inclusive se executado fora do Colab.


In [ ]:
# CÉLULA 1 — dependências do importador (somente ao executar)
%pip install -q "overturemaps>=2.1,<3" "geopandas>=1,<2" "shapely>=2,<3" "pyogrio>=0.10,<1"
print("Dependências para a Fase 9 disponíveis.")


In [ ]:
# CÉLULA 2 — importador com limite cumulativo +40, nenhum acesso externo
from pathlib import Path
IMPORTER_PATH=Path("/content/expand_overture_to_100_staging.py")
IMPORTER_SOURCE="#!/usr/bin/env python3\n\"\"\"Import Overture Places into private Supabase table, scoped to Brazil's municipalities.\n\nPre-req:\n    pip install overturemaps geopandas shapely pyogrio\n    export SUPABASE_URL='https://<project>.supabase.co'\n    export SUPABASE_SERVICE_ROLE_KEY='<service role secret>'\n    python scripts/expand_overture_to_100_staging.py --states MG --cities Lavras --database-max-mb 340\n\nPhase 9 top-up (staging only): accepts one state and 1-3 NEARBY cities per download.\nAll results are scanned; original GERS IDs/names remain unchanged. Only new rows\nare inserted up to 100 companies per municipality, stopping at 340 MiB. A\ngeographic bounding-box safety guard prevents oversized batched queries.\nNo automatic imports and no browser-side secrets.\n\"\"\"\nimport argparse\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport re\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport unicodedata\nfrom urllib import request, error\nfrom urllib.parse import urlparse, urlencode\n\nIBGE_2024_ROOT = (\n    \"https://geoftp.ibge.gov.br/organizacao_do_territorio/\"\n    \"malhas_territoriais/malhas_municipais/municipio_2024\"\n)\n\ndef state_boundaries_url(state):\n    \"\"\"IBGE offers one ZIP per UF; do not download the 199 MB national file.\"\"\"\n    if state not in STATE_CODES:\n        raise ValueError(\"Invalid Brazilian state code\")\n    return f\"{IBGE_2024_ROOT}/UFs/{state}/{state}_Municipios_2024.zip\"\n\ndef ensure_state_boundaries(state, source=None):\n    if source:\n        path = Path(source)\n        if not path.is_file():\n            raise RuntimeError(f\"IBGE boundaries file not found: {path}\")\n        return path\n    path = Path(\"data/ibge/\") / f\"{state}_Municipios_2024.zip\"\n    if not path.is_file():\n        path.parent.mkdir(parents=True, exist_ok=True)\n        url = state_boundaries_url(state)\n        print(f\"[{state}] Downloading IBGE municipal boundaries: {url}\", flush=True)\n        temp = path.with_suffix(\".partial\")\n        try:\n            request.urlretrieve(url, temp)\n            if temp.stat().st_size < 512:\n                raise RuntimeError(\"IBGE boundaries download appears empty\")\n            temp.rename(path)\n        finally:\n            temp.unlink(missing_ok=True)\n    return path\nSTATE_CODES = (\n    \"AC AL AP AM BA CE DF ES GO MA MT MS MG PA PB PR PE PI \"\n    \"RJ RN RS RO RR SC SP SE TO\"\n).split()\nBATCH_SIZE = 150\nSTAGING_URL = \"https://prlrxjothrslhswpkoza.supabase.co\"\nTARGET_PER_MUNICIPALITY = 100\nMAX_NEW_PER_MUNICIPALITY = 40\nSTORAGE_STOP_MIB = 340\n# Suppress named geographical features that are not potential companies.\nNON_LEAD_CATEGORIES = frozenset((\n    \"river\", \"lake\", \"stream\", \"waterfall\", \"mountain\", \"cave\",\n    \"island\", \"pond\", \"reservoir\", \"forest\", \"wetland\", \"beach\",\n    \"city\", \"town\", \"village\", \"municipality\", \"county\", \"state\",\n    \"country\", \"natural_feature\", \"neighborhood\", \"locality\",\n))\n\ndef is_lead_candidate(row):\n    return bool(row.get(\"name\") and\n                (row.get(\"category\") or \"\").lower() not in NON_LEAD_CATEGORIES)\n\n\ndef normalize(value):\n    \"\"\"Match the TypeScript search adapter's city/name normalization exactly.\"\"\"\n    letters = \"\".join(\n        c for c in unicodedata.normalize(\"NFD\", value.casefold())\n        if unicodedata.category(c) != \"Mn\"\n    )\n    return \" \".join(re.sub(r\"[^a-z0-9]+\", \" \", letters).split())\n\n\ndef first(values):\n    return next((v for v in (values or []) if isinstance(v, str) and v.strip()), None)\n\n\n# A social profile, WhatsApp contact or link shortener is NOT a verified\n# business website. This preserves truthful \"sem site\" prospecting filters.\nNON_WEBSITE_HOSTS = frozenset((\n    \"wa.me\", \"whatsapp.com\", \"api.whatsapp.com\", \"instagram.com\",\n    \"facebook.com\", \"fb.me\", \"m.me\", \"linktr.ee\", \"linktree.com\",\n    \"bit.ly\", \"tinyurl.com\", \"t.co\", \"youtube.com\", \"youtu.be\",\n    \"tiktok.com\", \"x.com\", \"twitter.com\", \"maps.google.com\",\n    \"goo.gl\", \"beacons.ai\",\n))\n\n\ndef is_business_website(value):\n    if not isinstance(value, str):\n        return False\n    try:\n        url = urlparse(value.strip())\n        host = (url.hostname or \"\").lower().removeprefix(\"www.\")\n        return (url.scheme in (\"http\", \"https\")\n                and bool(host) and \".\" in host\n                and host not in NON_WEBSITE_HOSTS\n                and not any(host.endswith(\".\" + blocked)\n                            for blocked in NON_WEBSITE_HOSTS))\n    except ValueError:\n        return False\n\n\ndef first_website(values):\n    return first([value for value in (values or []) if is_business_website(value)])\n\n\ndef select_diverse_pool(records, maximum):\n    \"\"\"Reserve 40% for contactable companies without identified websites.\n\n    Fill unused slots with the best available business contacts. Never promise\n    that a missing website is proof a business actually has no website.\n    \"\"\"\n    ranked = sorted(records, key=lead_quality, reverse=True)\n    no_site = [r for r in ranked if not r.get(\"website\")\n               and (r.get(\"phone\") or r.get(\"email\") or r.get(\"instagram\"))]\n    target_no_site = min(len(no_site), (maximum * 2 + 4) // 5)\n    chosen = no_site[:target_no_site]\n    selected = {r[\"id\"] for r in chosen}\n    for row in ranked:\n        if len(chosen) >= maximum:\n            break\n        if row[\"id\"] not in selected:\n            chosen.append(row)\n            selected.add(row[\"id\"])\n    return chosen\n\n\ndef api(base, key, path, method=\"GET\", payload=None, prefer=None):\n    data = json.dumps(payload, ensure_ascii=False).encode() if payload is not None else None\n    # Legacy service_role JWTs use Authorization; the newer sb_secret_ keys\n    # are opaque and must be sent only as apikey, never as Bearer JWT.\n    if not (key.startswith(\"eyJ\") or key.startswith(\"sb_secret_\")):\n        raise RuntimeError(\"Use somente a chave secreta de STAGING (sb_secret_ ou service_role legada).\")\n    headers = {\n        \"apikey\": key,\n        \"Content-Type\": \"application/json\",\n        \"Prefer\": prefer or \"resolution=merge-duplicates,return=minimal\",\n    }\n    if key.startswith(\"eyJ\"):\n        headers[\"Authorization\"] = f\"Bearer {key}\"\n    for attempt in range(5):\n        try:\n            req = request.Request(base + path, method=method, headers=headers, data=data)\n            with request.urlopen(req, timeout=45) as resp:\n                body = resp.read()\n                return json.loads(body) if body else None\n        except error.HTTPError as exc:\n            detail = exc.read(300).decode(\"utf-8\", errors=\"replace\")\n            if exc.code not in (429, 500, 502, 503, 504) or attempt == 4:\n                raise RuntimeError(f\"Supabase API {exc.code}: {detail}\") from exc\n        except (error.URLError, TimeoutError):\n            if attempt == 4:\n                raise\n        time.sleep(min(2 ** attempt, 15))\n    raise RuntimeError(\"Unreachable API retry state\")\n\n\ndef database_bytes(base, key):\n    result = api(base, key, \"/rest/v1/rpc/overture_storage_bytes\", method=\"POST\", payload={})\n    if not isinstance(result, int):\n        raise RuntimeError(\"Storage guard unavailable: run SQL migration first.\")\n    return result\n\n\ndef existing_city_places(base, key, state, city_key):\n    \"\"\"Read the already imported businesses of one municipality (max 100).\"\"\"\n    query = urlencode({\n        \"select\": \"id,name,name_key,latitude,longitude\",\n        \"state\": \"eq.\" + state, \"city_key\": \"eq.\" + city_key,\n        \"limit\": str(TARGET_PER_MUNICIPALITY + 1),\n    })\n    result = api(base, key, \"/rest/v1/overture_places?\" + query)\n    if not isinstance(result, list):\n        raise RuntimeError(\"Supabase did not return existing city businesses\")\n    if len(result) > TARGET_PER_MUNICIPALITY:\n        raise RuntimeError(\"Municipality exceeds 100 entries. Investigate before continuing.\")\n    return result\n\n\ndef original_phase9_count(base, key, state, city_key, current_count):\n    \"\"\"Preserve the FIRST before_count for lifetime +40, even on later sessions.\n\n    A missing checkpoint means this city has not been expanded by Phase 9.\n    The read is service_role only. Never reduce the initial baseline when the\n    company count rises; never re-import companies to reach the +40 cap.\n    \"\"\"\n    query = urlencode({\n        \"select\": \"before_count\",\n        \"state\": \"eq.\" + state, \"city_key\": \"eq.\" + city_key,\n        \"limit\": \"1\",\n    })\n    rows = api(base, key, \"/rest/v1/overture_phase9_attempts?\" + query)\n    if not isinstance(rows, list):\n        raise RuntimeError(\"Phase 9 baseline checkpoint could not be read\")\n    if not rows:\n        return current_count\n    first = rows[0].get(\"before_count\")\n    if not isinstance(first, int) or first < 0:\n        raise RuntimeError(\"Invalid Phase 9 baseline in staging\")\n    return min(current_count, first)\n\n\ndef same_business(a, b, meters=15.0):\n    \"\"\"Conservative alias check: EXACT normalized name AND location within 15m.\"\"\"\n    name_a = str(a.get(\"name_key\") or normalize(a.get(\"name\") or \"\"))\n    name_b = str(b.get(\"name_key\") or normalize(b.get(\"name\") or \"\"))\n    if not name_a or name_a != name_b:\n        return False\n    try:\n        la1, lo1 = float(a[\"latitude\"]), float(a[\"longitude\"])\n        la2, lo2 = float(b[\"latitude\"]), float(b[\"longitude\"])\n        if not all(math.isfinite(x) for x in (la1, lo1, la2, lo2)):\n            return False\n        x1, x2, delta_lon = math.radians(la1), math.radians(la2), math.radians(lo2-lo1)\n        delta_lat = x2-x1\n        value = math.sin(delta_lat/2)**2 + math.cos(x1)*math.cos(x2)*math.sin(delta_lon/2)**2\n        distance = 2*6371000*math.asin(min(1, math.sqrt(value)))\n        return distance <= meters\n    except (KeyError, ValueError, TypeError):\n        return False\n\n\ndef matches_any_existing(row, existing):\n    return any(same_business(row, prior) for prior in existing)\n\n\ndef only_new_within_cap(ranked, existing_ids, cap=TARGET_PER_MUNICIPALITY, existing_places=()):\n    \"\"\"Never select existing IDs or near-identical name+location duplicates.\"\"\"\n    room = max(0, cap - len(existing_ids))\n    if room == 0:\n        return []\n    selected, seen = [], set(existing_ids)\n    for row in ranked:\n        identifier = row.get(\"id\")\n        if not identifier or identifier in seen:\n            continue\n        if matches_any_existing(row, existing_places) or matches_any_existing(row, selected):\n            continue\n        seen.add(identifier)\n        selected.append(row)\n        if len(selected) >= room:\n            break\n    return selected\n\n\ndef offer_candidate(groups, row, max_per_group):\n    \"\"\"Bound memory while retaining NEW companies only, across entire input scan.\"\"\"\n    if any(prior.get(\"id\") == row[\"id\"] or same_business(prior, row)\n           for pool in groups.values() for prior in pool):\n        return False\n    pool = groups[\"with_site\" if row.get(\"website\") else \"without_site\"]\n    if len(pool) < max_per_group:\n        pool.append(row)\n        return True\n    worst = min(range(len(pool)), key=lambda idx: lead_quality(pool[idx]))\n    if lead_quality(row) > lead_quality(pool[worst]):\n        pool[worst] = row\n        return True\n    return False\n\n\ndef status(base, key, state, phase, count=0, message=None):\n    api(base, key, \"/rest/v1/overture_import_progress?on_conflict=state\", \"POST\", [{\n        \"state\": state, \"status\": phase,\n        \"imported_rows\": count, \"last_error\": message,\n        \"checked_at\": time.strftime(\"%Y-%m-%dT%H:%M:%SZ\", time.gmtime()),\n    }])\n\n\ndef municipalities_index(boundaries):\n    import geopandas as gpd\n\n    frame = gpd.read_file(\"zip://\" + str(boundaries.resolve()))\n    if frame.crs is None:\n        raise RuntimeError(\"The IBGE municipal boundary file has no CRS.\")\n    frame = frame.to_crs(epsg=4326).reset_index(drop=True)\n    columns = set(frame.columns)\n    if not {\"NM_MUN\", \"SIGLA_UF\"}.issubset(columns):\n        raise RuntimeError(f\"Unexpected IBGE municipalities columns: {sorted(columns)}\")\n    frame = frame[frame[\"SIGLA_UF\"].isin(STATE_CODES)].copy().reset_index(drop=True)\n    return frame\n\n\ndef place_record(feature, municipalities, state):\n    from shapely.geometry import Point\n\n    props = feature.get(\"properties\") or {}\n    coords = (feature.get(\"geometry\") or {}).get(\"coordinates\") or []\n    if len(coords) != 2:\n        return None\n    lon, lat = coords\n    if not isinstance(lon, (int, float)) or not isinstance(lat, (int, float)):\n        return None\n    name = (props.get(\"names\") or {}).get(\"primary\")\n    # GeoJSON exporters may place the Overture GERS id in Feature.id rather\n    # than properties.id. Preserve either representation.\n    identifier = props.get(\"id\") or feature.get(\"id\")\n    if not isinstance(name, str) or not name.strip() or not isinstance(identifier, str):\n        return None\n    if props.get(\"operating_status\") == \"permanently_closed\":\n        return None\n\n    # State bounding boxes overlap: map point to the true municipality.\n    point = Point(lon, lat)\n    matches = municipalities.sindex.query(point, predicate=\"intersects\")\n    if len(matches) == 0:\n        return None\n    mun = municipalities.iloc[int(matches[0])]\n    if mun[\"SIGLA_UF\"] != state:\n        return None\n    address = (props.get(\"addresses\") or [{}])[0]\n    if not isinstance(address, dict):\n        address = {}\n    taxonomy = props.get(\"taxonomy\") or {}\n    confidence = props.get(\"confidence\")\n    socials = props.get(\"socials\") or []\n    websites = props.get(\"websites\") or []\n    instagram = first([s for s in (socials + websites)\n                       if isinstance(s, str) and\n                       re.search(r\"https?://(?:www\\.)?instagram\\.com/\", s, re.I)])\n    return {\n        \"id\": identifier[:100],\n        \"name\": name.strip()[:240],\n        \"name_key\": normalize(name)[:240],\n        \"category\": str(taxonomy.get(\"primary\") or props.get(\"basic_category\") or \"\")[:120] or None,\n        \"city\": str(mun[\"NM_MUN\"])[:120],\n        \"city_key\": normalize(str(mun[\"NM_MUN\"]))[:120],\n        \"state\": str(mun[\"SIGLA_UF\"]),\n        \"address\": str(address.get(\"freeform\") or \"\")[:400] or None,\n        \"latitude\": lat, \"longitude\": lon,\n        \"phone\": (first(props.get(\"phones\")) or \"\")[:80] or None,\n        \"email\": (first(props.get(\"emails\")) or \"\")[:200] or None,\n        \"website\": (first_website(websites) or \"\")[:500] or None,\n        \"instagram\": instagram[:500] if instagram else None,\n        \"confidence\": confidence if isinstance(confidence, (int, float)) else None,\n        \"operating_status\": str(props.get(\"operating_status\") or \"\")[:35] or None,\n        \"data_release\": os.getenv(\"OVERTURE_DATA_RELEASE\", \"latest\")[:80],\n    }\n\n\ndef lead_quality(row):\n    \"\"\"Prioritize genuine, contactable establishments within the per-city quota.\"\"\"\n    confidence = row.get(\"confidence\")\n    weight = float(confidence) * 5 if isinstance(confidence, (int, float)) else 0\n    return (weight\n            + 12 * bool(row.get(\"phone\"))\n            + 11 * bool(row.get(\"email\"))\n            + 7 * bool(row.get(\"website\"))\n            + 3 * bool(row.get(\"instagram\"))\n            + 2 * bool(row.get(\"category\")))\n\n\ndef batch_bbox_allowed(bounds, city_count, max_span=1.8, max_area=0.9):\n    \"\"\"Prevent multi-town downloads that accidentally cover most of a state.\"\"\"\n    if city_count <= 1:\n        return True\n    west, south, east, north = [float(v) for v in bounds]\n    width, height = east-west, north-south\n    return (width > 0 and height > 0 and width <= max_span\n            and height <= max_span and width*height <= max_area)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\"--states\", default=\"\", help=\"Exactly one Brazilian UF\")\n    parser.add_argument(\"--cities\", default=\"\", help=\"One to three nearby municipalities in the same state\")\n    parser.add_argument(\"--boundaries\", default=None, help=\"Optional pre-existing official IBGE ZIP\")\n    parser.add_argument(\"--database-max-mb\", type=int, default=STORAGE_STOP_MIB)\n    parser.add_argument(\"--max-records-per-city\", type=int, default=TARGET_PER_MUNICIPALITY)\n    parser.add_argument(\"--dry-run\", action=\"store_true\")\n    args = parser.parse_args()\n    if not 30 <= args.database_max_mb <= STORAGE_STOP_MIB:\n        parser.error(\"Database safety limit must be <= 340 MiB\")\n    if args.max_records_per_city != TARGET_PER_MUNICIPALITY:\n        parser.error(\"Final cap must be exactly 100 per municipality\")\n    states = [s.strip().upper() for s in args.states.split(\",\") if s.strip()]\n    city_keys = [normalize(s.strip()) for s in args.cities.split(\",\") if s.strip()]\n    if len(states) != 1 or states[0] not in STATE_CODES:\n        parser.error(\"One valid state is required\")\n    if len(city_keys) < 1 or len(city_keys) > 3 or len(set(city_keys)) != len(city_keys):\n        parser.error(\"Provide 1-3 distinct nearby municipalities\")\n    state = states[0]\n    base = os.getenv(\"SUPABASE_URL\", \"\").rstrip(\"/\")\n    secret = os.getenv(\"SUPABASE_SERVICE_ROLE_KEY\", \"\")\n    if not args.dry_run and (base != STAGING_URL or\n                            not (secret.startswith(\"eyJ\") or secret.startswith(\"sb_secret_\"))):\n        parser.error(\"Only the Januário Leads STAGING project with a server-side secret is accepted\")\n\n    boundaries = ensure_state_boundaries(state, args.boundaries)\n    municipalities = municipalities_index(boundaries)\n    state_munis = municipalities.loc[\n        (municipalities.SIGLA_UF == state) &\n        (municipalities.NM_MUN.map(lambda name: normalize(str(name)) in city_keys))\n    ].copy()\n    found = {normalize(str(name)) for name in state_munis.NM_MUN}\n    if set(city_keys) != found:\n        raise RuntimeError(\"Municipality not found in IBGE boundaries\")\n    west, south, east, north = [float(v) for v in state_munis.total_bounds]\n    if not batch_bbox_allowed((west, south, east, north), len(city_keys)):\n        raise RuntimeError(\"BATCH_TOO_WIDE: municipality polygons are too dispersed for one download\")\n    bbox = \",\".join(f\"{v:.6f}\" for v in (west-.005, south-.005, east+.005, north+.005))\n    print(f\"[{state}] Overture bbox {bbox}; cities={len(city_keys)}\", flush=True)\n    if args.dry_run:\n        return 0\n\n    limit_bytes = args.database_max_mb * 1024 * 1024\n    storage_before = database_bytes(base, secret)\n    if storage_before >= limit_bytes:\n        print(\"STOP: database safety threshold\", flush=True)\n        return 2\n    existing = {city: existing_city_places(base, secret, state, city)\n                for city in city_keys}\n    # Add at most FORTY NEW companies per city, even if it originally held\n    # fewer than 60. Preserve the first baseline across Colab resumptions.\n    baseline = {city: original_phase9_count(base, secret, state, city, len(rows))\n                for city, rows in existing.items()}\n    allowance = {\n        city: min(TARGET_PER_MUNICIPALITY - len(rows),\n                  MAX_NEW_PER_MUNICIPALITY - max(0, len(rows) - baseline[city]))\n        for city, rows in existing.items()\n    }\n    targets = {city: rows for city, rows in existing.items()\n               if allowance[city] > 0}\n    for key in city_keys:\n        if key not in targets:\n            print(f\"ALREADY_COMPLETE: {state}/{key}\", flush=True)\n    if not targets:\n        return 0\n\n    with tempfile.TemporaryDirectory(prefix=\"overture-phase9-\") as temp:\n        output = Path(temp) / f\"{state}.geojsonseq\"\n        subprocess.run([\"overturemaps\", \"download\", f\"--bbox={bbox}\",\n                        \"--type=place\", \"-f\", \"geojsonseq\", \"-o\", str(output)],\n                       check=True)\n        # Scan all downloaded places ONCE, route by the official IBGE polygon.\n        # Every bounded pool holds ONLY businesses not already in Supabase.\n        groups_by_city = {key: {\"with_site\": [], \"without_site\": []} for key in targets}\n        known_ids = {key: {r[\"id\"] for r in rows} for key,rows in targets.items()}\n        stats = {key: {\"scanned\": 0, \"eligible\": 0, \"existing\": 0}\n                 for key in targets}\n        with output.open(\"r\", encoding=\"utf-8\") as handle:\n            for line in handle:\n                line = line.lstrip(\"\\x1e\").strip()\n                if not line:\n                    continue\n                row = place_record(json.loads(line), state_munis, state)\n                if row is None:\n                    continue\n                key = row[\"city_key\"]\n                if key not in targets or not is_lead_candidate(row):\n                    continue\n                stats[key][\"scanned\"] += 1\n                if row[\"id\"] in known_ids[key] or matches_any_existing(row, targets[key]):\n                    stats[key][\"existing\"] += 1\n                    continue\n                stats[key][\"eligible\"] += 1\n                offer_candidate(groups_by_city[key], row, TARGET_PER_MUNICIPALITY)\n\n        total_inserted = 0\n        for key in city_keys:\n            if key not in targets:\n                continue\n            if database_bytes(base, secret) >= limit_bytes:\n                print(\"STOP: database safety threshold\", flush=True)\n                return 2\n            prior = targets[key]\n            capacity = allowance[key]\n            groups = groups_by_city[key]\n            ranked = select_diverse_pool(groups[\"with_site\"]+groups[\"without_site\"],capacity)\n            fresh = existing_city_places(base, secret, state, key)\n            # A concurrent importer cannot exceed the original city baseline\n            # +40; re-check the current row count before inserting.\n            max_total = min(TARGET_PER_MUNICIPALITY,\n                            baseline[key] + MAX_NEW_PER_MUNICIPALITY)\n            proposed = only_new_within_cap(ranked,{r[\"id\"] for r in fresh},\n                                           cap=max_total, existing_places=fresh)\n            if proposed:\n                api(base,secret,\"/rest/v1/overture_places?on_conflict=id\",\"POST\",proposed,\n                    prefer=\"resolution=ignore-duplicates,return=minimal\")\n            verified = existing_city_places(base, secret, state, key)\n            gained = max(0,len(verified)-len(prior))\n            total_inserted += gained\n            print(\"PHASE9_CITY_RESULT=\"+json.dumps({\n                \"state\":state,\"city_key\":key,\"before\":len(prior),\n                \"after\":len(verified),\"gained\":gained,\n                \"new_candidates\":stats[key][\"eligible\"],\n                \"status\":\"gained\" if gained else \"no_more\"\n            },ensure_ascii=False),flush=True)\n            if len(verified)>TARGET_PER_MUNICIPALITY:\n                raise RuntimeError(\"A municipality exceeded the safety cap of 100\")\n        print(f\"BATCH_SUMMARY: state={state}; downloaded_for={len(targets)}; \"\n              f\"new_companies={total_inserted}; \"\n              f\"database_mib={database_bytes(base,secret)/1048576:.2f}\",flush=True)\n    return 0\n\n\nif __name__ == \"__main__\":\n    sys.exit(main())\n"
IMPORTER_PATH.write_text(IMPORTER_SOURCE,encoding="utf-8")
compile(IMPORTER_SOURCE,str(IMPORTER_PATH),"exec")
print("Importador Overture fase 9 salvo localmente e sintaxe validada. Nenhuma importação.")


In [ ]:
# CÉLULA 3 — configurações e testes locais; ZERO consultas externas
import datetime as dt
import getpass
import json
import math
import os
import re
import subprocess
import random
import sys
import time
import unicodedata
from urllib.parse import urlencode
from urllib.request import Request,urlopen
from urllib.error import HTTPError

STAGING_URL="https://prlrxjothrslhswpkoza.supabase.co"
TARGET_PER_CITY=100
MAX_NEW_PER_CITY=40
MAX_MUNICIPIOS_SESSAO=5572
MAX_DOWNLOADS_SESSAO=240  # até 720 municípios com agrupamento de 3; Colab pode interromper antes
MAX_CITIES_PER_DOWNLOAD=3
BATCH_NEARBY_KM=25
MIN_INTERVAL_BETWEEN_DOWNLOADS=75  # segundos entre INÍCIOS de downloads
DOWNLOAD_JITTER_MAX_SECONDS=15
GLOBAL_RATE_COOLDOWN_HOURS=6  # após erro 429 da fonte: respeitar a limitação entre sessões
STORAGE_STOP_MIB=340
TIME_LIMIT_MINUTES=540
MAX_ERRORS=12
MAX_CONSECUTIVE_ERRORS=3
MAX_DIFFERENT_UFS_WITH_FAILURES=3
PAGE_SIZE=1000
PAUSE_SECONDS=0
TRANSIENT_RETRY_SECONDS=40
FAILURE_BACKOFF_SECONDS=(4,12)
MIB=1024*1024
assert TARGET_PER_CITY==100 and MAX_NEW_PER_CITY==40 and STORAGE_STOP_MIB==340
assert MAX_CITIES_PER_DOWNLOAD==3 and BATCH_NEARBY_KM<=35
assert MIN_INTERVAL_BETWEEN_DOWNLOADS>=60 and GLOBAL_RATE_COOLDOWN_HOURS>=3
assert 1<=MAX_MUNICIPIOS_SESSAO<=5572 and 30<=TIME_LIMIT_MINUTES<=600
assert IMPORTER_PATH.is_file() and IMPORTER_SOURCE

def staged_rest(secret,path,method="GET",body=None):
    if not path.startswith("/rest/v1/") or path.startswith("/rest/v1//"):
        raise ValueError("Apenas rotas PostgREST são permitidas")
    headers={"apikey":secret,"Accept":"application/json","Content-Type":"application/json"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    data=json.dumps(body).encode("utf-8") if body is not None else None
    req=Request(STAGING_URL+path,method=method,data=data,headers=headers)
    with urlopen(req,timeout=45) as response:
        raw=response.read(2000000)
        return json.loads(raw.decode("utf-8")) if raw else None

def storage_mib(secret):
    raw=staged_rest(secret,"/rest/v1/rpc/overture_storage_bytes","POST",{})
    if not isinstance(raw,int):raise RuntimeError("Falha no controle de armazenamento")
    return raw/MIB

def municipality_rows(secret):
    out=[]
    offset=0
    while True:
        qs=urlencode({"select":"state,city,city_key,imported_rows,center_lat,center_lon",
                      "order":"state.asc,city_key.asc",
                      "limit":str(PAGE_SIZE),"offset":str(offset)})
        page=staged_rest(secret,"/rest/v1/overture_phase9_city_centers?"+qs)
        if not isinstance(page,list):raise RuntimeError("Não foi possível listar municípios")
        out.extend(page)
        if len(page)<PAGE_SIZE:break
        offset+=len(page)
    return out

def pending_cities(secret):
    """Read ONLY the Phase 9 queue, never the Phase 1–7 retry schedule."""
    now=dt.datetime.now(dt.timezone.utc).isoformat()
    result=set()
    offset=0
    while True:
        qs=urlencode({"select":"state,city_key","retry_after":"gt."+now,
                      "order":"state.asc,city_key.asc","limit":str(PAGE_SIZE),
                      "offset":str(offset)})
        page=staged_rest(secret,"/rest/v1/overture_phase9_attempts?"+qs)
        if not isinstance(page,list):raise RuntimeError("Falha ao verificar retomadas da Fase 9")
        result.update((r["state"],r["city_key"]) for r in page)
        if len(page)<PAGE_SIZE:break
        offset+=len(page)
    return result

def phase9_baselines(secret):
    """Initial counts from prior attempts (service-role staging only)."""
    result={}
    offset=0
    while True:
        qs=urlencode({"select":"state,city_key,before_count",
                      "order":"state.asc,city_key.asc",
                      "limit":str(PAGE_SIZE),"offset":str(offset)})
        page=staged_rest(secret,"/rest/v1/overture_phase9_attempts?"+qs)
        if not isinstance(page,list):
            raise RuntimeError("Falha ao carregar o baseline da Fase 9")
        for row in page:
            count=row.get("before_count")
            if not isinstance(count,int) or count<0:
                raise RuntimeError("Baseline de municipio invalido")
            key=(row["state"],row["city_key"])
            result[key]=min(result.get(key,count),count)
        if len(page)<PAGE_SIZE:break
        offset+=len(page)
    return result

def sorted_targets(rows,pending=(),tried=(),baselines=None):
    """Balance UFs so a few very large states do not monopolize the session."""
    blocked=set(pending)|set(tried)
    buckets={}
    for r in rows:
        if not isinstance(r,dict):continue
        n=r.get("imported_rows")
        uf=r.get("state")
        if (not isinstance(n,int) or n<=0 or n>=TARGET_PER_CITY
                or not uf or not r.get("city") or not r.get("city_key")
                or (uf,r["city_key"]) in blocked):continue
        initial=baselines.get((uf,r["city_key"]),n) if baselines is not None else n
        if n>=min(TARGET_PER_CITY,initial+MAX_NEW_PER_CITY):continue
        buckets.setdefault(uf,[]).append(r)
    for uf in buckets:
        buckets[uf].sort(key=lambda r:(-r["imported_rows"],r["city_key"]))
    result=[]
    # Round robin across states, favoring towns at the old 60-company cap.
    states=sorted(buckets)
    for i in range(max((len(v) for v in buckets.values()),default=0)):
        for uf in states:
            if i<len(buckets[uf]):result.append(buckets[uf][i])
    return result

def distance_km(a,b):
    """Centroid-distance heuristic; importer's polygon bbox guard is final."""
    try:
        lat1,lon1=float(a["center_lat"]),float(a["center_lon"])
        lat2,lon2=float(b["center_lat"]),float(b["center_lon"])
        if not all(math.isfinite(x) for x in (lat1,lon1,lat2,lon2)):
            return float("inf")
        x1,x2=math.radians(lat1),math.radians(lat2)
        dy=x2-x1
        dx=math.radians(lon2-lon1)
        h=math.sin(dy/2)**2+math.cos(x1)*math.cos(x2)*math.sin(dx/2)**2
        return 12742.0*math.asin(min(1,math.sqrt(h)))
    except (KeyError,ValueError,TypeError):
        return float("inf")

def nearby_batches(targets,max_cities=MAX_CITIES_PER_DOWNLOAD,max_distance=BATCH_NEARBY_KM):
    """Share a download ONLY among close municipalities of the same UF."""
    available={(r["state"],r["city_key"]) for r in targets}
    by_state={}
    for row in targets:
        by_state.setdefault(row["state"],[]).append(row)
    batches=[]
    for anchor in targets:
        akey=(anchor["state"],anchor["city_key"])
        if akey not in available:continue
        available.remove(akey)
        group=[anchor]
        if max_cities>1:
            for candidate in by_state[anchor["state"]]:
                ckey=(candidate["state"],candidate["city_key"])
                if ckey not in available:continue
                if all(distance_km(candidate,current)<=max_distance for current in group):
                    group.append(candidate)
                    available.remove(ckey)
                    if len(group)==max_cities:break
        batches.append(group)
    return batches

def parse_time(iso):
    return dt.datetime.fromisoformat(str(iso).replace("Z","+00:00")).astimezone(dt.timezone.utc)

def latest_attempt(secret,filter_code=None):
    query={"select":"attempted_at","order":"attempted_at.desc","limit":"1"}
    if filter_code:query["error_code"]="eq."+filter_code
    rows=staged_rest(secret,"/rest/v1/overture_phase9_attempts?"+urlencode(query))
    return parse_time(rows[0]["attempted_at"]) if rows else None

def rate_cooldown_remaining(now,last_rate):
    if not last_rate:return 0
    elapsed=(now-last_rate).total_seconds()
    return max(0,int(GLOBAL_RATE_COOLDOWN_HOURS*3600-elapsed))

def spacing_remaining(now,last_attempt):
    if not last_attempt:return 0
    return max(0,MIN_INTERVAL_BETWEEN_DOWNLOADS-(now-last_attempt).total_seconds())

def record_attempt(secret,state,city,city_key,reason,before,after,error_code=None):
    """Private checkpoint, never mutates any Overture company or national progress."""
    if reason not in ("gained","no_more","error","partial"):
        raise ValueError("Invalid Phase 9 checkpoint status")
    from datetime import timedelta
    delay={"gained":7,"no_more":14,"error":0.25,"partial":0.25}[reason]
    due=(dt.datetime.now(dt.timezone.utc)+timedelta(days=delay)).isoformat()
    if error_code is not None and not re.fullmatch(r"[a-z0-9_]{1,80}",error_code):
        raise ValueError("Código de erro fora do padrão seguro")
    # Never overwrite the first observed before_count: the +40 cap must
    # continue to apply after any resumed Colab session.
    query=urlencode({"select":"before_count","state":"eq."+state,
                    "city_key":"eq."+city_key,"limit":"1"})
    previous=staged_rest(secret,"/rest/v1/overture_phase9_attempts?"+query)
    if not isinstance(previous,list):
        raise RuntimeError("Nao foi possivel ler baseline de checkpoint")
    baseline=before
    if previous:
        old=previous[0].get("before_count")
        if not isinstance(old,int) or old<0:
            raise RuntimeError("Baseline historico invalido")
        baseline=min(before,old)
    payload={"state":state,"city":city,"city_key":city_key,
             "result":reason,"before_count":baseline,"after_count":after,
             "added_count":max(0,after-baseline),"retry_after":due,
             "error_code":error_code}
    headers={"apikey":secret,"Accept":"application/json","Content-Type":"application/json",
             "Prefer":"resolution=merge-duplicates,return=minimal"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    req=Request(STAGING_URL+"/rest/v1/overture_phase9_attempts?on_conflict=state,city_key",
                method="POST",data=json.dumps(payload).encode("utf-8"),headers=headers)
    with urlopen(req,timeout=35) as resp:resp.read(20000)

def actual_city_count(secret,state,city_key):
    query=urlencode({"select":"id","state":"eq."+state,"city_key":"eq."+city_key})
    headers={"apikey":secret,"Accept":"application/json","Prefer":"count=exact"}
    if secret.startswith("eyJ"):headers["Authorization"]="Bearer "+secret
    req=Request(STAGING_URL+"/rest/v1/overture_places?"+query,headers=headers,method="HEAD")
    with urlopen(req,timeout=45) as resp:
        number=(resp.headers.get("Content-Range") or "").rsplit("/",1)[-1]
        if not number.isdigit():raise RuntimeError("Não foi possível confirmar a contagem")
        return int(number)

# Classifica SOMENTE a categoria técnica. Nunca imprime stdout/stderr bruto,
# URLs, endpoints com apiKey, tokens, corpo de resposta, nem traceback completo.
def classify_import_error(output,returncode=None,timed_out=False):
    if timed_out:
        return "timeout_subprocesso"
    content=str(output or "").lower()[-12000:]
    if "batch_too_wide" in content:
        return "batch_largura"
    if "database safety threshold" in content or "database reached configured" in content:
        return "limite_banco"
    if "supabase api 401" in content or "supabase api 403" in content or "jwt expired" in content:
        return "supabase_auth"
    if "supabase api 429" in content:
        return "supabase_taxa"
    if any(s in content for s in ("http error 429","http 429","too many requests","throttl","rate limit","http 429")):
        return "fonte_taxa"
    if any(s in content for s in ("http error 403","http error 401","accessdenied","access denied","forbidden")):
        return "fonte_acesso"
    if any(s in content for s in ("http error 503","http error 502","http error 504","service unavailable","bad gateway","gateway timeout")):
        return "fonte_indisponivel"
    if "supabase api" in content and any(s in content for s in (" 500"," 502"," 503"," 504")):
        return "supabase_indisponivel"
    if any(s in content for s in ("no module named","modulenotfounderror","importerror","command not found","filenotfounderror")):
        return "dependencia_ausente"
    if any(s in content for s in ("out of memory","memoryerror","cannot allocate memory","killed")):
        return "sem_memoria"
    if any(s in content for s in ("certificate verify failed","sslerror","tls error")):
        return "tls"
    if any(s in content for s in ("timed out","timeouterror","readtimeout","connecttimeout")):
        return "rede_timeout"
    if any(s in content for s in ("connection reset","connection refused","network is unreachable","name resolution","dns","urlerror")):
        return "rede_conexao"
    if any(s in content for s in ("municipality not found","unexpected ibge","ibge archive does not cover","boundaries download")):
        return "malha_ibge"
    if any(s in content for s in ("jsondecodeerror","invalid json","valueerror")):
        return "dados_invalidos"
    if "calledprocesserror" in content or "overturemaps download" in content:
        return "download_overture"
    if "supabase api" in content:
        return "supabase_http"
    if returncode is not None and returncode<0:
        return "processo_sinal"
    return "falha_desconhecida"

def stop_on_global_error(code):
    return code in ("limite_banco","supabase_auth","fonte_acesso","supabase_taxa",
                    "fonte_taxa","dependencia_ausente","sem_memoria","tls")

def transient_error(code):
    return code in ("rede_timeout","rede_conexao","fonte_indisponivel",
                    "supabase_indisponivel","timeout_subprocesso")

def failure_backoff(streak):
    return FAILURE_BACKOFF_SECONDS[min(max(streak-1,0),len(FAILURE_BACKOFF_SECONDS)-1)]

# Testes que não dependem de chave nem de acesso à internet.
sample_rows=[
    {"state":"MG","city":"Lavras","city_key":"lavras","imported_rows":60},
    {"state":"SP","city":"Santos","city_key":"santos","imported_rows":100},
    {"state":"RJ","city":"Niterói","city_key":"niteroi","imported_rows":22},
]
assert [r["city_key"] for r in sorted_targets(sample_rows)]==["lavras","niteroi"]
assert [r["city_key"] for r in sorted_targets(sample_rows+[{"state":"MG","city":"Alfenas","city_key":"alfenas","imported_rows":59}])]==["lavras","niteroi","alfenas"]
assert [r["city_key"] for r in sorted_targets(sample_rows,pending={("MG","lavras")})]==["niteroi"]
assert TARGET_PER_CITY-60==40 and MAX_NEW_PER_CITY==40 and TARGET_PER_CITY-100==0
assert not sorted_targets([{"state":"MG","city":"Exemplo","city_key":"exemplo","imported_rows":70}],
    baselines={("MG","exemplo"):30})
assert classify_import_error("Supabase API 401: secret_here")== "supabase_auth"
assert classify_import_error("HTTP Error 429: throttled")=="fonte_taxa"
assert classify_import_error("connection reset by peer")=="rede_conexao"
assert classify_import_error("private url",returncode=1)=="falha_desconhecida"
assert classify_import_error("",timed_out=True)=="timeout_subprocesso"
assert stop_on_global_error("fonte_taxa") and not stop_on_global_error("rede_timeout")
assert transient_error("rede_timeout") and failure_backoff(3)==12
assert classify_import_error("BATCH_TOO_WIDE polygons")=="batch_largura"
assert distance_km({"center_lat":-21,"center_lon":-45},{"center_lat":-21.01,"center_lon":-45})<2
tgroup=[
 {"state":"MG","city":"Lavras","city_key":"lavras","center_lat":-21.24,"center_lon":-45.00},
 {"state":"SP","city":"Santos","city_key":"santos","center_lat":-23.96,"center_lon":-46.33},
 {"state":"MG","city":"Nepomuceno","city_key":"nepomuceno","center_lat":-21.24,"center_lon":-45.03},
 {"state":"MG","city":"Juiz de Fora","city_key":"juiz de fora","center_lat":-21.74,"center_lon":-43.34},
]
assert [[r["city_key"] for r in b] for b in nearby_batches(tgroup)]==[
 ["lavras","nepomuceno"],["santos"],["juiz de fora"]]
assert rate_cooldown_remaining(dt.datetime(2026,11,13,20,tzinfo=dt.timezone.utc),
    dt.datetime(2026,11,13,19,tzinfo=dt.timezone.utc))==18000
assert spacing_remaining(dt.datetime(2026,11,13,20,tzinfo=dt.timezone.utc),
    dt.datetime(2026,11,13,20,tzinfo=dt.timezone.utc))==75
print("TESTES OFFLINE OK: meta 100, retomada, diagnóstico e proteção de erros.")


In [ ]:
# CÉLULA 4 — expansão MANUAL com lotes geográficos e respeito ao limite de requisições
def ampliar_100():
    from collections import deque
    print("FASE 9 V4 — OVERTURE: ate +40 empresas novas, total ate 100 por municipio.")
    print("Proteção: até",MAX_CITIES_PER_DOWNLOAD,"municípios próximos por download;",
          "intervalo mínimo:",MIN_INTERVAL_BETWEEN_DOWNLOADS,"segundos.")
    print("Parada de armazenamento:",STORAGE_STOP_MIB,"MiB.")
    confirm=input("Para INICIAR somente na HOMOLOGAÇÃO, digite AMPLIAR_100: ").strip()
    if confirm!="AMPLIAR_100":
        print("CANCELADO: nenhuma importação.")
        return
    secret=getpass.getpass("Chave SECRETA da HOMOLOGAÇÃO (campo oculto): ").strip()
    if not (secret.startswith("sb_secret_") or secret.startswith("eyJ")):
        print("Credencial ausente ou inválida; nenhuma importação.")
        return
    imported=0; improved=0; completed=0; without_new=0
    attempts=0; downloads=0; grouped_downloads=0; extra_retries=0
    errors=0; consecutive_errors=0
    failures_by_category={}; recent_failed_states=[]
    pause_reason=None; deferred_count=0
    try:
        now=dt.datetime.now(dt.timezone.utc)
        rate_checkpoint=latest_attempt(secret,"fonte_taxa")
        cooldown=rate_cooldown_remaining(now,rate_checkpoint)
        if cooldown>0:
            until=rate_checkpoint+dt.timedelta(hours=GLOBAL_RATE_COOLDOWN_HOURS)
            print("PARADA SEGURA: Overture informou limite de requisições.")
            print("Não consultar a fonte antes de:",until.astimezone().isoformat(timespec="minutes"))
            print("Restam aproximadamente",int(math.ceil(cooldown/60)),"minutos do período de segurança.")
            return
        used=storage_mib(secret)
        print("Banco de homologação:",round(used,2),"MiB")
        if used>=STORAGE_STOP_MIB:
            print("PARADA: proteção de armazenamento ativa.")
            return
        last_attempt=latest_attempt(secret)
        if last_attempt:
            until=last_attempt+dt.timedelta(seconds=MIN_INTERVAL_BETWEEN_DOWNLOADS)
            if dt.datetime.now(dt.timezone.utc)<until:
                wait=(until-dt.datetime.now(dt.timezone.utc)).total_seconds()
                print("Aguardando espaçamento entre sessões:",math.ceil(wait),"segundos.")
                time.sleep(max(0,wait))
        rows=municipality_rows(secret)
        pending=pending_cities(secret)
        baselines=phase9_baselines(secret)
        deferred_count=len(pending)
        targets=sorted_targets(rows,pending,baselines=baselines)
        queue=deque(nearby_batches(targets))
        print("Municípios na malha:",len(rows),"| elegíveis agora:",len(targets),
              "| temporariamente adiados:",deferred_count)
        print("Grupos de download estimados:",len(queue))
        deadline=time.monotonic()+TIME_LIMIT_MINUTES*60
        next_download_start=time.monotonic()
        while queue and attempts<MAX_MUNICIPIOS_SESSAO and downloads<MAX_DOWNLOADS_SESSAO:
            if errors>=MAX_ERRORS:
                pause_reason="teto_erros_sessao"
                break
            if time.monotonic()>deadline-180:
                pause_reason="tempo_sessao"
                break
            if storage_mib(secret)>=STORAGE_STOP_MIB:
                pause_reason="teto_banco"
                break
            group=queue.popleft()
            # Max municipalities is a cap, even when selecting a batch.
            if len(group)>MAX_MUNICIPIOS_SESSAO-attempts:
                group=group[:MAX_MUNICIPIOS_SESSAO-attempts]
            if not group:break
            state=group[0]["state"]
            cities=[r["city"] for r in group]
            before={}
            for row in group:
                before[row["city_key"]]=actual_city_count(secret,state,row["city_key"])
            group=[r for r in group if before[r["city_key"]]<TARGET_PER_CITY]
            if not group:continue
            cities=[r["city"] for r in group]
            # Respect the interval across STARTS (even after short or failed downloads).
            delay=max(0,next_download_start-time.monotonic())
            if delay:
                if time.monotonic()+delay>deadline-180:
                    pause_reason="tempo_sessao"
                    break
                print("ESPERA ENTRE DOWNLOADS:",round(delay),"segundos.")
                time.sleep(delay)
            env=os.environ.copy()
            env["SUPABASE_URL"]=STAGING_URL
            env["SUPABASE_SERVICE_ROLE_KEY"]=secret
            command=[sys.executable,str(IMPORTER_PATH),
                     "--states",state,"--cities",",".join(cities),
                     "--max-records-per-city",str(TARGET_PER_CITY),
                     "--database-max-mb",str(STORAGE_STOP_MIB)]
            run=None; error_code=None; attempts_in_run=0
            for trial in range(2):
                if downloads>=MAX_DOWNLOADS_SESSAO:
                    pause_reason="limite_downloads_sessao"
                    break
                delay=max(0,next_download_start-time.monotonic())
                if delay:
                    if time.monotonic()+delay>deadline-180:
                        pause_reason="tempo_sessao"
                        break
                    time.sleep(delay)
                if time.monotonic()>deadline-180:
                    pause_reason="tempo_sessao"
                    break
                downloads+=1
                attempts_in_run+=1
                next_download_start=time.monotonic()+MIN_INTERVAL_BETWEEN_DOWNLOADS+random.uniform(0,DOWNLOAD_JITTER_MAX_SECONDS)
                print("DOWNLOAD",downloads,"| UF:",state,
                      "| cidades:",", ".join(cities),"| grupo:",len(group),flush=True)
                try:
                    run=subprocess.run(command,env=env,text=True,capture_output=True,
                                       timeout=min(1200,max(90,int(deadline-time.monotonic()-120))))
                    stdout=(run.stdout or "")+"\n"+(run.stderr or "")
                    error_code=(classify_import_error(stdout,run.returncode)
                                if run.returncode!=0 else None)
                except subprocess.TimeoutExpired:
                    run=None
                    error_code="timeout_subprocesso"
                if error_code is None:
                    break
                if error_code=="batch_largura" and len(group)>1:
                    # Too much geographic spread: retry each separately later.
                    # No network download took place (importer's bbox preflight).
                    for single in reversed(group):
                        queue.appendleft([single])
                    print("DESAGRUPADO:",len(group),
                          "municípios dispersos. Serão consultados individualmente.")
                    break
                if trial==0 and transient_error(error_code) and time.monotonic()+MIN_INTERVAL_BETWEEN_DOWNLOADS+120<deadline:
                    extra_retries+=1
                    print("REDE TRANSITÓRIA:",error_code,
                          "| uma nova tentativa somente após intervalo controlado")
                    continue
                break
            if pause_reason:break
            if error_code=="batch_largura" and len(group)>1:
                continue
            attempts+=len(group)
            if len(group)>1 and error_code is None:grouped_downloads+=1
            for row in group:
                key=row["city_key"]
                before_n=before[key]
                after_n=actual_city_count(secret,state,key)
                gained=max(0,after_n-before_n)
                if after_n>100:
                    pause_reason="municipio_maior_100"
                    print("PARADA: município ultrapassou 100, investigar.")
                    break
                imported+=gained
                if gained:improved+=1
                if after_n>=100 and before_n<100:completed+=1
                if gained:
                    print("SALVO:",row["city"]+"/"+state,"| +",gained,
                          "| agora:",after_n,"/100")
                elif error_code is None:
                    without_new+=1
                    print("SEM NOVAS:",row["city"]+"/"+state)
                reason=("partial" if error_code and gained else "error" if error_code
                        else "gained" if gained else "no_more")
                try:
                    record_attempt(secret,state,row["city"],key,reason,
                                   before_n,after_n,error_code)
                except Exception as exc:
                    errors+=1
                    pause_reason="checkpoint_indisponivel"
                    print("PARADA: falha ao registrar progresso:",type(exc).__name__)
                    break
            if pause_reason:break
            if error_code is not None:
                errors+=1
                consecutive_errors+=1
                failures_by_category[error_code]=failures_by_category.get(error_code,0)+1
                recent_failed_states=(recent_failed_states+[state])[-5:]
                print("ERRO PROTEGIDO:",error_code,"| UF:",state,
                      "| sequência:",consecutive_errors)
                if stop_on_global_error(error_code):
                    pause_reason="erro_global_"+error_code
                    if error_code in ("fonte_taxa","supabase_taxa"):
                        print("LIMITE DA FONTE: aguarde o período de segurança antes de uma nova sessão.")
                    break
                if consecutive_errors>=MAX_CONSECUTIVE_ERRORS:
                    pause_reason="falhas_consecutivas"
                    break
                if len(set(recent_failed_states))>=MAX_DIFFERENT_UFS_WITH_FAILURES:
                    pause_reason="falhas_ufs_distintas"
                    break
                delay=failure_backoff(consecutive_errors)
                print("PAUSA DE RECUPERAÇÃO:",delay,"segundos.")
                time.sleep(delay)
            else:
                consecutive_errors=0
                recent_failed_states=[]
        print("\n"+"="*58)
        print("RELATÓRIO FASE 9 V4 — OVERTURE +40 / ATE 100")
        print("Municípios tentados nesta sessão:",attempts)
        print("Municípios que ganharam pelo menos 1 empresa:",improved)
        print("Municípios que chegaram a 100:",completed)
        print("Empresas inéditas confirmadas por contagem:",imported)
        print("Municípios sem novos dados:",without_new)
        print("Downloads executados:",downloads)
        print("Downloads aproveitados por múltiplos municípios:",grouped_downloads)
        print("Novas tentativas por falhas transitórias:",extra_retries)
        print("Municípios adiados por checkpoints:",deferred_count)
        print("Categorias de erro:",dict(sorted(failures_by_category.items())))
        print("Erros:",errors,"| Motivo da parada:",pause_reason or "fila_ou_limite_sessao")
        print("Banco ao final:",round(storage_mib(secret),2),"MiB")
        print("Sem chamadas ao Geoapify ou Google Places; credenciais não registradas.")
        print("="*58)
    except HTTPError as exc:
        print("FALHA HTTP no Supabase:",exc.code,"| nenhuma chave registrada.")
    finally:
        secret=""
        print("Credenciais descartadas. Desconecte o runtime ao terminar.")

ampliar_100()
